In [27]:
class Environment:
    def __init__ (self, graph):
        self.graph = graph

    def get_percept(self, node):
        return node

    def bfs_search(self, start, goal):
        visited = []
        queue = []
        visited.append(start)
        queue.append(start)
        while queue:
            node = queue.pop(0)
            print(f"Visiting: {node}")
            if node == goal:
                return f"Goal {goal} found"
            for neighbor in self.graph.get(node, []):
                if neighbor not in visited:
                    visited.append(neighbor)
                    queue.append(neighbor)
        return "Goal not found.."

    def dfs_search(self, start, goal):
        visited = []
        stack = []
        visited.append(start)
        stack.append(start)
        while stack:
            node = stack.pop()
            print(f"Visiting {node}")
            if node == goal:
                return f"Goal {goal} found.."
            for neighbor in reversed(self.graph.get(node, [])):
                 if neighbor not in visited:
                    visited.append(neighbor)
                    stack.append(neighbor)
        return "Goal not found.."

class goal_based_bfs:
    def __init__ (self, goal):
        self.goal = goal

    def formulate_goal(self, percept):
        if percept == self.goal:
            return "Goal reached"
        return "Searching..."

    def act(self, percept, envirnoment):
        goal_status = self.formulate_goal(percept)
        if goal_status == "Goal reached":
            return f"Goal {self.goal} found"
        else:
            return environment.bfs_search(percept, self.goal)

class goal_based_dfs:
    def __init__ (self, goal):
        self.goal = goal

    def formulate_goal(self, percept):
        if percept == self.goal:
            return "Goal reached"
        return "Searching..."

    def act(self, percept, envirnoment):
        goal_status = self.formulate_goal(percept)
        if goal_status == "Goal reached":
            return f"Goal {self.goal} found"
        else:
            return environment.dfs_search(percept, self.goal)


def run_agent(agent, environment, start_node):
    percept = environment.get_percept(start_node)
    action = agent.act(percept, environment)
    print(action)
            

In [33]:
tree = {
    0: [1, 3],
    1: [0, 3],
    2: [4,5],
    3: [0, 1, 6, 4],
    4: [3, 2, 5],
    5: [4, 2, 6],
    6: [3, 5]
}

start_node = 0
goal_node = 5

agent = goal_based_bfs(goal_node)
agent2 = goal_based_dfs(goal_node)
environment = Environment(tree)
print("BFS:")
run_agent(agent, environment, start_node)
print("\n\nDFS:")
run_agent(agent2, environment, start_node)

BFS:
Visiting: 0
Visiting: 1
Visiting: 3
Visiting: 6
Visiting: 4
Visiting: 5
Goal 5 found


DFS:
Visiting 0
Visiting 1
Visiting 3
Visiting 6
Visiting 5
Goal 5 found..


## Genetic Algorithm

In [42]:
import random


population_size = 10
mutation_rate = 0.1
n = 8

#defining fitness function
def fitness_function(individual):
    non_attacking_pairs = 0
    total_pairs = n * (n-1) // 2

    #no conflict check
    for i in range(n):
        for j in range(i+1, n):
            if individual[i] != individual[j] and abs(individual[i] - individual[j]) != abs(i-j):
                non_attacking_pairs += 1
    return non_attacking_pairs/total_pairs

#step 2: initialize the population
def create_random_individuals():
    return random.sample(range(n), n)

#selelecting parents
def select_parent(population, fitness_score):
    sorted_population = [board for _, board in sorted(zip(fitness_score, population), reverse = True)]
    return sorted_population

def crossover(parent1, parent2):
    point = random.randint(1, n-2)
    child = parent1[:point] + parent2[point:]

    missing = set(range(n)) - set(child)
    duplicates = [col for col in child if child.count(col) > 1]
    for i in range(len(child)):
        if child.count(child[i]) > 1:
            child[i] = missing.pop()
    return child

#mutation
def mutation(individual):
    idx1, idx2 = random.sample(range(n), 2)
    individual[idx1], individual[idx2] = individual[idx2], individual[idx1]
    return individual

#genetic algo
def genetic_algorithm():
    population = [create_random_individuals() for _ in range(population_size)]
    generation = 0
    best_fitness = 0

    while best_fitness < 1.0 and generation < 100:
        fitness_scores = [fitness_function(ind) for ind in population]
        best_fitness = max(fitness_scores)
        print(f"Generation {generation} Best fitness {best_fitness}")

        if best_fitness == 1.0:
            break

        #select parent
        parents = select_parent(population, fitness_scores)

        #crossover
        new_population = [crossover(random.choice(parents), random.choice(parents)) for _ in range(population_size)]

        #mutation
        for i in range(len(new_population)):
            if random.random() < mutation_rate:
                new_population[i] = mutation(new_population[i])

        population = new_population
        generation += 1


    best_individual = max(population, key=fitness_function)
    return best_individual, fitness_function(best_individual)


solution, fitness = genetic_algorithm()
print(f"Best solution: {solution}\nBest fitness: {fitness}")
    

Generation 0 Best fitness 0.9285714285714286
Generation 1 Best fitness 0.8928571428571429
Generation 2 Best fitness 0.9285714285714286
Generation 3 Best fitness 0.8928571428571429
Generation 4 Best fitness 0.8928571428571429
Generation 5 Best fitness 0.8928571428571429
Generation 6 Best fitness 0.9285714285714286
Generation 7 Best fitness 0.9285714285714286
Generation 8 Best fitness 0.9285714285714286
Generation 9 Best fitness 0.8571428571428571
Generation 10 Best fitness 0.9285714285714286
Generation 11 Best fitness 0.9285714285714286
Generation 12 Best fitness 0.8571428571428571
Generation 13 Best fitness 0.8928571428571429
Generation 14 Best fitness 0.8571428571428571
Generation 15 Best fitness 0.8928571428571429
Generation 16 Best fitness 0.8928571428571429
Generation 17 Best fitness 0.8928571428571429
Generation 18 Best fitness 0.8928571428571429
Generation 19 Best fitness 0.9642857142857143
Generation 20 Best fitness 0.8571428571428571
Generation 21 Best fitness 0.857142857142857

## A* Search

In [52]:
def a_star(graph, start, goal):
    frontier = [(start, 0 + heuristic[start])]
    visited = set()
    cum_cost = {start:0}
    came_from = {start:None}

    while frontier:
        frontier.sort(key=lambda x:x[1])
        current_node, curren_f = frontier.pop(0)
        if current_node in visited:
            continue
        print(current_node, end = " ")
        visited.add(current_node)

        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = came_from[current_node]
            path.reverse()
            print(f"\nPath found: {path}")
            return
        for neighbor, cost in graph[current_node].items():
            new_cum_cost = cum_cost[current_node] + cost
            f_cost = new_cum_cost + heuristic[neighbor]
            if neighbor not in cum_cost or new_cum_cost < cum_cost[neighbor]:
                cum_cost[neighbor] = new_cum_cost
                came_from[neighbor] = current_node
                frontier.append((neighbor, f_cost))
    print("Goal not found")



graph = {
    'A': {'B': 2, 'C': 1},
    'B': {'D': 4, 'E': 3},
    'C': {'F': 1, 'G': 5},
    'D': {'H': 2},
    'E': {},
    'F': {'I': 6},
    'G': {},
    'H': {},
    'I': {}
}

# Heuristic function (estimated cost to reach goal 'I')
heuristic = {
    'A': 7,
    'B': 6,
    'C': 5,
    'D': 4,
    'E': 7,
    'F': 3,
    'G': 6,
    'H': 2,
    'I': 0 # Goal node
}

print("\nFollowing is the A* Search:")
a_star(graph, 'A', 'I')


Following is the A* Search:
A C F B I 
Path found: ['A', 'C', 'F', 'I']


In [56]:
from queue import PriorityQueue

# Example graph represented as an adjacency list with heuristic values included
graph = {
    'A': [('B', 5, 9), ('C', 8, 5)], # (neighbor, cost, heuristic)
    'B': [('D', 10, 4)], # (neighbor, cost, heuristic)
    'C': [('E', 3, 7)], # (neighbor, cost, heuristic)
    'D': [('F', 7, 5)], # (neighbor, cost, heuristic)
    'E': [('F', 2, 1)], # (neighbor, cost, heuristic)
    'F': [] # (neighbor, cost, heuristic)
}

def astar_search(graph, start, goal):
    visited = set() # Set to keep track of visited nodes
    pq = PriorityQueue() # Priority queue to prioritize nodes based on f-value (cost + heuristic)
    pq.put((0, start)) # Enqueue the start node with priority 0
    while not pq.empty():
        cost, node = pq.get() # Dequeue the node with the lowest priority
        if node not in visited:
            print(node, end=' ') # Print the current node
            visited.add(node) # Mark the current node as visited
        if node == goal: # Check if the goal node is reached
            print("\nGoal reached!")
            return True

        for neighbor, edge_cost, heuristic in graph[node]: # Explore neighbors of the current node
            if neighbor not in visited:
                # Calculate f-value for the neighbor (cost + heuristic)
                f_value = cost + edge_cost + heuristic
                pq.put((f_value, neighbor)) # Enqueue neighbor with priority based on f-value
    print("\nGoal not reachable!")
    return False

# Example usage:
print("A* Search Path:")
astar_search(graph, 'A', 'F')

A* Search Path:
A C B E F 
Goal reached!


True

## Task 1: BFS and DFS for Grid Search
Problem:
Find a path from 'P' (starting position) to 'T' (target) in a grid using Breadth-First Search (BFS) and Depth-First Search (DFS).

In [58]:
import random

# Define the grid
grid = [
    ['O', 'O', 'X', 'O', 'T'],
    ['O', 'X', 'O', 'O', 'X'],
    ['P', 'O', 'O', 'X', 'O'],
    ['X', 'X', 'O', 'O', 'O'],
    ['O', 'O', 'O', 'X', 'O']
]

directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def find_positions(grid):
    start = target = None
    for i in range(len(grid)):
        for j in range(len(grid[0])):
            if grid[i][j] == 'P':
                start = (i, j)
            elif grid[i][j] == 'T':
                target = (i, j)
    return start, target

# BFS using a list as a queue
def bfs(grid):
    start, target = find_positions(grid)
    queue = [(start, [start])]
    visited = set([start])

    while queue:
        (x, y), path = queue.pop(0)  # Manual queue (FIFO)
        if (x, y) == target:
            return path

        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if 0 <= nx < len(grid) and 0 <= ny < len(grid[0]) and grid[nx][ny] != 'X' and (nx, ny) not in visited:
                queue.append(((nx, ny), path + [(nx, ny)]))
                visited.add((nx, ny))
    return None

# DFS using a list as a stack
def dfs(grid):
    start, target = find_positions(grid)
    stack = [(start, [start])]
    visited = set([start])

    while stack:
        (x, y), path = stack.pop()  # Manual stack (LIFO)
        if (x, y) == target:
            return path

        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if 0 <= nx < len(grid) and 0 <= ny < len(grid[0]) and grid[nx][ny] != 'X' and (nx, ny) not in visited:
                stack.append(((nx, ny), path + [(nx, ny)]))
                visited.add((nx, ny))
    return None

# Running searches
print("BFS Path:", bfs(grid))
print("DFS Path:", dfs(grid))


BFS Path: [(2, 0), (2, 1), (2, 2), (1, 2), (1, 3), (0, 3), (0, 4)]
DFS Path: [(2, 0), (2, 1), (2, 2), (1, 2), (1, 3), (0, 3), (0, 4)]


In [60]:
# Grid representation
grid = [
    ['O', 'O', 'X', 'O', 'T'],
    ['O', 'X', 'O', 'O', 'X'],
    ['P', 'O', 'O', 'X', 'O'],
    ['X', 'X', 'O', 'O', 'O'],
    ['O', 'O', 'O', 'X', 'O']
]

# Directions: up, down, left, right
directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

# Helper function to check if a position is valid
def is_valid(x, y, visited):
    return 0 <= x < len(grid) and 0 <= y < len(grid[0]) and grid[x][y] != 'X' and not visited[x][y]

# BFS Implementation
def bfs():
    start = None
    target = None

    # Find start ('P') and target ('T')
    for i in range(len(grid)):
        for j in range(len(grid[0])):
            if grid[i][j] == 'P':
                start = (i, j)
            elif grid[i][j] == 'T':
                target = (i, j)

    if not start or not target:
        return "Start or Target not found!"

    queue = [(start[0], start[1], [])]  # (x, y, path)
    visited = [[False for _ in range(len(grid[0]))] for _ in range(len(grid))]

    while queue:
        x, y, path = queue.pop(0)
        if (x, y) == target:
            return path + [(x, y)]  # Return the complete path

        visited[x][y] = True

        # Explore neighbors
        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if is_valid(nx, ny, visited):
                queue.append((nx, ny, path + [(x, y)]))

    return "No path found!"

# DFS Implementation
def dfs():
    start = None
    target = None

    # Find start ('P') and target ('T')
    for i in range(len(grid)):
        for j in range(len(grid[0])):
            if grid[i][j] == 'P':
                start = (i, j)
            elif grid[i][j] == 'T':
                target = (i, j)

    if not start or not target:
        return "Start or Target not found!"

    stack = [(start[0], start[1], [])]  # (x, y, path)
    visited = [[False for _ in range(len(grid[0]))] for _ in range(len(grid))]

    while stack:
        x, y, path = stack.pop()
        if (x, y) == target:
            return path + [(x, y)]  # Return the complete path

        visited[x][y] = True

        # Explore neighbors
        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if is_valid(nx, ny, visited):
                stack.append((nx, ny, path + [(x, y)]))

    return "No path found!"

# Run BFS and DFS
print("BFS Path:", bfs())
print("DFS Path:", dfs())

BFS Path: [(2, 0), (2, 1), (2, 2), (1, 2), (1, 3), (0, 3), (0, 4)]
DFS Path: [(2, 0), (2, 1), (2, 2), (1, 2), (1, 3), (0, 3), (0, 4)]


## Task 2: A Search Algorithm *
Problem:
Find the optimal path in a weighted grid using the A* algorithion:

In [62]:
import random

grid = [
    [1, 2, 3, '#', 4],
    [1, '#', 1, 2, 2],
    [2, 3, 1, '#', 1],
    ['#', '#', 2, 1, 1],
    [1, 1, 2, 2, 1]
]

directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def a_star(grid):
    N, M = len(grid), len(grid[0])
    start, target = (0, 0), (N - 1, M - 1)
    queue = [(0, start, [start])]  # (cost, position, path)
    g_cost = {start: 0}

    while queue:
        queue.sort()  # Manual sorting instead of priority queue
        cost, (x, y), path = queue.pop(0)

        if (x, y) == target:
            return path

        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if 0 <= nx < N and 0 <= ny < M and grid[nx][ny] != '#':
                new_cost = g_cost[(x, y)] + grid[nx][ny]
                if (nx, ny) not in g_cost or new_cost < g_cost[(nx, ny)]:
                    g_cost[(nx, ny)] = new_cost
                    f_cost = new_cost + heuristic((nx, ny), target)
                    queue.append((f_cost, (nx, ny), path + [(nx, ny)]))

    return None

print("Optimal Path:", a_star(grid))


Optimal Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (3, 2), (3, 3), (3, 4), (4, 4)]


In [64]:
# Weighted grid representation
grid = [
    [1, 2, 3, '#', 4],
    [1, '#', 1, 2, 2],
    [2, 3, 1, '#', 1],
    ['#', '#', 2, 1, 1],
    [1, 1, 2, 2, 1]
]

# Directions: up, down, left, right
directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

# Helper function to check if a position is valid
def is_valid_a_star(x, y, visited):
    return 0 <= x < len(grid) and 0 <= y < len(grid[0]) and grid[x][y] != '#' and not visited[x][y]

# Heuristic function (Manhattan distance)
def heuristic(x, y, target_x, target_y):
    return abs(x - target_x) + abs(y - target_y)

# A* Search Implementation
def a_star():
    start = (0, 0)
    target = (len(grid) - 1, len(grid[0]) - 1)

    open_list = [(0, start[0], start[1], [])]  # (f_score, x, y, path)
    visited = [[False for _ in range(len(grid[0]))] for _ in range(len(grid))]

    while open_list:
        # Sort open list by f_score
        open_list.sort(key=lambda x: x[0])
        f_score, x, y, path = open_list.pop(0)

        if (x, y) == target:
            return path + [(x, y)]  # Return the complete path

        visited[x][y] = True

        # Explore neighbors
        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if is_valid_a_star(nx, ny, visited):
                g_score = len(path) + 1  # Cost so far
                h_score = heuristic(nx, ny, target[0], target[1])
                f_score = g_score + h_score
                open_list.append((f_score, nx, ny, path + [(x, y)]))

    return "No path found!"

# Run A* Search
print("A* Path:", a_star())

A* Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (4, 2), (4, 3), (4, 4)]


## Task 3: Genetic Algorithm Optimization
Problem:
Optimize f(x()=^ )2
 −1 within x∈[0,31] using a genetic algorithm.

In [69]:
import random

# Fitness function: f(x) = 2x^2 - 1
def fitness(binary_string):
    x = int(binary_string, 2)  # Convert binary to integer
    return 2 * (x ** 2) - 1

# Generate a random binary string (6-bit)
def random_individual():
    return ''.join(str(random.randint(0, 1)) for _ in range(6))

# Tournament Selection
def tournament_selection(population):
    sample = [population[random.randint(0, len(population) - 1)] for _ in range(3)]
    return max(sample, key=fitness)

# Uniform Crossover
def uniform_crossover(parent1, parent2):
    child = ''.join(parent1[i] if random.randint(0, 1) else parent2[i] for i in range(len(parent1)))
    return child

# Adaptive Mutation
def adaptive_mutation(individual, generation, max_generations):
    mutation_rate = 0.1 + (0.9 * (1 - generation / max_generations))
    mutated = ''.join(
        str(1 - int(bit)) if random.random() < mutation_rate else bit for bit in individual
    )
    return mutated

# Genetic Algorithm
def genetic_algorithm(generations=100, population_size=10):
    population = [random_individual() for _ in range(population_size)]
    
    for generation in range(generations):
        new_population = []
        for _ in range(population_size):
            parent1 = tournament_selection(population)
            parent2 = tournament_selection(population)
            child = uniform_crossover(parent1, parent2)
            child = adaptive_mutation(child, generation, generations)
            new_population.append(child)
        
        population = new_population

    best = max(population, key=fitness)
    return best, fitness(best)

# Running the Genetic Algorithm
best_solution, best_fitness = genetic_algorithm()
print(f"Best Solution: {best_solution}, Fitness: {best_fitness}")


Best Solution: 111110, Fitness: 7687


In [71]:
import random

# Function to evaluate fitness
def fitness(individual):
    x = int(individual, 2)  # Convert binary string to integer
    return 2 * (x ** 2) - 1

# Generate initial population
def generate_population(pop_size, chromosome_length):
    return [''.join(random.choice('01') for _ in range(chromosome_length)) for _ in range(pop_size)]

# Tournament selection
def tournament_selection(population, fitnesses, tournament_size=3):
    selected = random.sample(list(zip(population, fitnesses)), tournament_size)
    return max(selected, key=lambda x: x[1])[0]

# Uniform crossover
def uniform_crossover(parent1, parent2):
    child1, child2 = '', ''
    for i in range(len(parent1)):
        if random.random() < 0.5:
            child1 += parent1[i]
            child2 += parent2[i]
        else:
            child1 += parent2[i]
            child2 += parent1[i]
    return child1, child2

# Mutation with adaptive probability
def mutate(individual, mutation_rate):
    mutated = ''
    for bit in individual:
        if random.random() < mutation_rate:
            mutated += '1' if bit == '0' else '0'
        else:
            mutated += bit
    return mutated

# Genetic Algorithm
def genetic_algorithm(pop_size=10, chromosome_length=6, generations=50, mutation_rate=0.05):
    population = generate_population(pop_size, chromosome_length)
    best_solution = None
    best_fitness = float('-inf')

    for generation in range(generations):
        fitnesses = [fitness(ind) for ind in population]

        # Update best solution
        for ind, fit in zip(population, fitnesses):
            if fit > best_fitness:
                best_fitness = fit
                best_solution = ind

        # Create next generation
        new_population = []
        while len(new_population) < pop_size:
            parent1 = tournament_selection(population, fitnesses)
            parent2 = tournament_selection(population, fitnesses)
            child1, child2 = uniform_crossover(parent1, parent2)
            new_population.append(mutate(child1, mutation_rate))
            new_population.append(mutate(child2, mutation_rate))

        population = new_population[:pop_size]

    return best_solution, best_fitness

# Run Genetic Algorithm
best_solution, best_fitness = genetic_algorithm()
print(f"Best Solution: {int(best_solution, 2)}, Fitness: {best_fitness}")

Best Solution: 63, Fitness: 7937
